# OpenCV Crash Course: 2027 Edition
## Module 8: Segmentation (YOLO-seg & SAM)
**Estimated time: 15 minutes**

- YOLO-seg: exact object shape, not just a box
- Use the mask with OpenCV (area, background blur)
- SAM: segment anything from one click, no training

> Masks are needed to measure area, fill level, defects, or cut out an object.

In [ ]:
# !pip install ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request, os
from ultralytics import YOLO, SAM

def show_grid(images, titles, figsize=(16, 6)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    for ax, img, t in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap='gray')
        ax.set_title(t, fontsize=11)
        ax.axis('off')
    plt.show()

os.makedirs('input', exist_ok=True)
if not os.path.exists('input/bus.jpg'):
    urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', 'input/bus.jpg')

## 1. YOLO-seg

Same API as detection, just a `-seg` model. `r.masks.xy` gives polygons in image coordinates.

In [ ]:
seg = YOLO('models/yolo26n-seg.pt')
img = cv2.imread('input/bus.jpg')
r = seg(img, verbose=False)[0]
h, w = img.shape[:2]

people = np.zeros((h, w), np.uint8)
for poly, c in zip(r.masks.xy, r.boxes.cls.int().tolist()):
    if seg.names[c] == 'person':
        cv2.fillPoly(people, [poly.astype(np.int32)], 255)
print(f"People cover {100 * cv2.countNonZero(people) / (h * w):.1f}% of the image")

## 2. Use the Mask with OpenCV

In [ ]:
blurred  = cv2.GaussianBlur(img, (51, 51), 0)
portrait = np.where(people[..., None] == 255, img, blurred)
show_grid([r.plot(), people, portrait], ['YOLO-seg', 'People mask', 'Background blur'])

## 3. SAM: Segment Anything

Click on any object (`points`) or draw a rough box (`bboxes`). No training, no class list.
Great for auto-labelling datasets for Module 7.

In [ ]:
sam = SAM('models/sam2.1_t.pt')
res = sam(img, points=[[400, 500]], labels=[1], verbose=False)[0]

vis = res.plot()
cv2.circle(vis, (400, 500), 12, (0, 0, 255), -1)
mask = cv2.resize(res.masks.data[0].cpu().numpy().astype(np.uint8) * 255, (w, h))
show_grid([vis, mask], ['SAM: one click (red dot)', 'Mask'], figsize=(14, 7))

---
## Summary

| Task | Code |
|------|------|
| Segment | `YOLO('models/yolo26n-seg.pt')(img)[0]` |
| Polygons | `r.masks.xy` |
| Polygon to mask | `cv2.fillPoly(mask, [poly.astype(np.int32)], 255)` |
| SAM click | `SAM('models/sam2.1_t.pt')(img, points=[[x, y]], labels=[1])` |
| SAM box | `sam(img, bboxes=[[x1, y1, x2, y2]])` |

**Next:** Module 9, OCR